In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio
import time

In [2]:
load_dotenv(override=True)

True

In [3]:
contacts = [
    {"name": "John Smith", "email": "kaczynskanatalia0@gmail.com", "company": "TechCorp"},
    {"name": "Anna Nowak", "email": "kaczynska.natalia@o2.pl", "company": "InnovateCo"},
]

# TEST
print("Contacts:", contacts)

Contacts: [{'name': 'John Smith', 'email': 'kaczynskanatalia0@gmail.com', 'company': 'TechCorp'}, {'name': 'Anna Nowak', 'email': 'kaczynska.natalia@o2.pl', 'company': 'InnovateCo'}]


In [4]:
@function_tool
def get_contact_list() -> list:
    """Get the list of sales prospects with their name, email and company"""
    return contacts

In [5]:
@function_tool
def send_bulk_html_email(subject: str, html_body_template: str) -> Dict[str, str]:
    """
    Send personalized HTML emails to all contacts in the list.
    The html_body_template should contain {name} and {company} placeholders.
    """

    contacts_local = contacts
    results = []
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("natalia.kaczynska.programista@gmail.com")

    for i, contact in enumerate(contacts_local):
        personalized_body = html_body_template.replace("{name}", contact["name"])
        personalized_body = personalized_body.replace("{company}", contact["company"])

        to_email = To(contact["email"])
        content = Content("text/html", personalized_body)
        mail = Mail(from_email, to_email, subject, content).get()

        try:
            response = sg.client.mail.send.post(request_body=mail)
            results.append({
                "contact": contact["name"],
                "email": contact["email"],
                "status": "success",
                "code": response.status_code
            })

            if i < len(contacts) - 1:
                time.sleep(3)

        except Exception as e:
            results.append({
                "contact": contact["name"],
                "email": contact["email"],
                "status": "failed",
                "error": str(e)
            })

    return {
        "total_contacts": len(contacts),
        "sent": len([r for r in results if r["status"] == "success"]),
        "failed": len([r for r in results if r["status"] == "failed"]),
        "details": results
    }

In [ ]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

sales_agent1 = Agent(
    name="Professional Sales Agent",
    instructions=instructions1,
    model="gpt-4o-mini"
)

sales_agent2 = Agent(
    name="Engaging Sales Agent",
    instructions=instructions2,
    model="gpt-4o-mini"
)

sales_agent3 = Agent(
    name="Busy Sales Agent",
    instructions=instructions3,
    model="gpt-4o-mini"
)

In [6]:
campaign_instructions = """
    You are a Campaign Manager responsible for sending bulk email campaigns.

    Your job:
    1. Use get_contact_list to see how many contacts we have
    2. Prepare an html email that includes {name} and {company} placeholdes
    3. Use send_bulk_html_email to send personalized emails to all contacts
    4. Report how many emails were successfully sent

    Important: The email body MUST include {name} and {company} placeholders for personalization.
"""

campaign_manager = Agent(
    name="Campaign Manager",
    instructions=campaign_instructions,
    tools=[get_contact_list, send_bulk_html_email],
    model="gpt-4o-mini",
    handoff_description="Send bulk personalized email campaign to all contacts"
)

message = "Send a simple campaign email introducing ComplAI to all contacts"

with trace("Test Campaign Manager"):
    result = await Runner.run(campaign_manager, message)
    
print("Result:", result.final_output)

Result: The email campaign introducing ComplAI has been successfully sent to all contacts. Here are the details:

- **Total Contacts**: 2
- **Emails Sent**: 2
- **Emails Failed**: 0

All recipients received their personalized emails:
1. **John Smith** - kaczynskanatalia0@gmail.com
2. **Anna Nowak** - kaczynska.natalia@o2.pl

If you need anything else, feel free to ask!
